# 03. Первичный пространственный анализ

**ЛР 2, шаг 8.** Цель: получить первые ответы на вопросы заказчика и проверить, что выбранные данные и методы работают.

Аналитические вопросы взяты из устава и требований:

1. **Сколько школ и поликлиник в каждом районе Уфы и какова их плотность** — на км² и на 10 тыс. жителей?
2. **Насколько школы удалены от поликлиник**: какая доля школ имеет поликлинику ближе 1 км и в каких районах школа дальше всего от медицинской помощи?

Все расстояния и площади считаются в метрической проекции UTM (EPSG:32640), а не в градусах. Данные — очищенные слои из \`data/processed/\` (ноутбук 02), © участники OpenStreetMap.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BariNiaz/pm-lab-2-schoolmed-access/blob/main/notebooks/03_spatial_analysis.ipynb)

## Настройка среды

Ноутбук работает в **Google Colab** и локально. В Colab ячейка ниже ставит библиотеки и подключает Google Диск, локально — работает с папками репозитория.

Перед первым запуском впишите в \`PROJECT_FOLDER\` имя папки группы на Google Диске. Запуск: **Среда выполнения → Выполнить все**.

In [ ]:
# === Настройка среды: запустите эту ячейку первой ===
import sys, subprocess
from pathlib import Path

PROJECT_FOLDER = "pm-lab-2-schoolmed-access"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas>=1.0", "folium", "mapclassify"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    if not BASE.exists():
        raise FileNotFoundError(
            f"На Google Диске нет папки «{PROJECT_FOLDER}». Создайте её или добавьте ярлык "
            "общей папки группы в «Мой диск» (приложение Б, шаг 2) и запустите ячейку снова.")
else:
    BASE = Path("..")   # локальный запуск из папки notebooks/ репозитория

RAW, MANUAL, PROCESSED = BASE / "data" / "raw", BASE / "data" / "manual", BASE / "data" / "processed"
IMG, MAPS = BASE / "docs" / "img", BASE / "docs" / "maps"
for folder in (RAW, MANUAL, PROCESSED, IMG, MAPS):
    folder.mkdir(parents=True, exist_ok=True)
print("Среда:", "Google Colab" if IN_COLAB else "локальная", "· рабочая папка:", BASE.resolve())

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

districts = gpd.read_file(PROCESSED / "districts.gpkg")
schools = gpd.read_file(PROCESSED / "schools.gpkg")
clinics = gpd.read_file(PROCESSED / "clinics.gpkg")
boundary = gpd.read_file(PROCESSED / "boundary.gpkg")
pop = pd.read_csv(MANUAL / "population_by_district.csv")

METRIC_CRS = boundary.estimate_utm_crs()
print("Метрическая проекция:", METRIC_CRS)
print("Районов:", len(districts), "· школ:", len(schools), "· поликлиник:", len(clinics))
print("Население в таблице:", len(pop), "районов, год:", int(pop["year"].max()))

## Вопрос 1. Сколько школ и поликлиник в каждом районе и какова плотность

Метод: площади районов считаются в метрической проекции; объекты относятся к району пространственным объединением «точка внутри района» (\`sjoin\`, условие \`within\`). Плотность — на км² и на 10 тыс. жителей.

In [ ]:
districts_m = districts.to_crs(METRIC_CRS).copy()
districts_m["area_km2"] = districts_m.area / 1e6

schools_m = schools.to_crs(METRIC_CRS)
clinics_m = clinics.to_crs(METRIC_CRS)

district_names = districts_m[["name", "geometry"]].rename(columns={"name": "district"})

s_join = gpd.sjoin(schools_m, district_names, how="left", predicate="within")
c_join = gpd.sjoin(clinics_m, district_names, how="left", predicate="within")

schools_count = s_join.groupby("district").size().rename("школ")
clinics_count = c_join.groupby("district").size().rename("поликлиник")

q1 = districts_m[["name", "area_km2"]].set_index("name")
q1 = q1.join(schools_count).join(clinics_count).fillna(0)
q1[["школ", "поликлиник"]] = q1[["школ", "поликлиник"]].astype(int)
q1 = q1.join(pop.set_index("district")[["population"]], how="left")
q1["школ на 10 тыс."] = (q1["школ"] / q1["population"] * 10000).round(1)
q1["поликлиник на 10 тыс."] = (q1["поликлиник"] / q1["population"] * 10000).round(1)
q1["школ на км²"] = (q1["школ"] / q1["area_km2"]).round(2)
q1["поликлиник на км²"] = (q1["поликлиник"] / q1["area_km2"]).round(2)
q1["area_km2"] = q1["area_km2"].round(1)

q1 = q1.sort_values("поликлиник на 10 тыс.")
pd.set_option("display.width", 200)
print(q1.to_string())
print("\nВсего: школ", int(q1["школ"].sum()), "· поликлиник", int(q1["поликлиник"].sum()),
      "· площадь", round(q1["area_km2"].sum(), 1), "км²")
print("В среднем по городу: школ на 10 тыс.", round(q1["школ"].sum() / q1["population"].sum() * 10000, 1),
      "· поликлиник на 10 тыс.", round(q1["поликлиник"].sum() / q1["population"].sum() * 10000, 1))

q1.to_csv(PROCESSED / "districts_metrics.csv")

## Вопрос 2. Насколько школы удалены от поликлиник

Метод: для каждой школы ищется ближайшая поликлиника (\`sjoin_nearest\`, расстояние в метрах в UTM). Затем по районам считаются медианное расстояние и доля школ, у которых поликлиника ближе 1 км. Это оценка по прямой, а не по пешеходной сети, — ограничение метода записано в выводах.

In [ ]:
nearest = gpd.sjoin_nearest(
    schools_m[["osm_type", "osm_id", "name", "geometry"]].rename(columns={"name": "school"}),
    clinics_m[["name", "geometry"]].rename(columns={"name": "clinic"}),
    how="left", max_distance=None, distance_col="dist_m",
).reset_index()
nearest = nearest[~nearest["index"].duplicated()]

schools_meta = s_join[["district"]]
nearest = nearest.join(schools_meta, on="index")
nearest["within_1km"] = nearest["dist_m"] <= 1000

q2 = nearest.groupby("district").agg(
    школ=("osm_id", "count"),
    медиана_м=("dist_m", "median"),
    доля_в_1км=("within_1km", "mean"),
)
q2["медиана_м"] = q2["медиана_м"].round(0).astype(int)
q2["доля_в_1км"] = (q2["доля_в_1км"] * 100).round(1)
q2 = q2.sort_values("доля_в_1км")

print("По районам: сколько школ, медианное расстояние до ближайшей поликлиники и доля школ с поликлиникой в 1 км:")
print(q2.to_string())
print("\nПо городу: медианное расстояние", int(nearest["dist_m"].median()), "м;",
      "доля школ с поликлиникой ближе 1 км:", round(nearest["within_1km"].mean() * 100, 1), "%")
worst = nearest.loc[nearest["dist_m"].idxmax()]
print("Самая удалённая школа:", worst["school"], "—", worst["district"], "—", int(worst["dist_m"]), "м до поликлиники")

nearest.to_csv(PROCESSED / "schools_nearest_clinic.csv", index=False)

## Карты

Статичные карты для отчёта — \`docs/img/\`, интерактивная карта для заказчика — \`docs/maps/\`. На обеих указан источник данных.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

q1_plot = districts_m.set_index("name").join(q1[["поликлиник на 10 тыс."]])
q1_plot.plot(ax=axes[0], column="поликлиник на 10 тыс.", cmap="YlOrRd", legend=True,
             edgecolor="white", legend_kwds={"label": "Поликлиник на 10 тыс. жителей", "shrink": 0.7})
schools_m.plot(ax=axes[0], color="#1f6feb", markersize=6)
axes[0].set_title("Школы и поликлиники на 10 тыс. жителей\n(© участники OpenStreetMap)", fontsize=12)
axes[0].set_axis_off()

q1_plot2 = districts_m.set_index("name").join(q2[["доля_в_1км"]])
q1_plot2.plot(ax=axes[1], column="доля_в_1км", cmap="RdYlGn", legend=True,
              edgecolor="white", legend_kwds={"label": "Доля школ с поликлиникой в 1 км, %", "shrink": 0.7})
clinics_m.plot(ax=axes[1], color="#d1242f", markersize=6)
axes[1].set_title("Поликлиники и доля школ с поликлиникой в 1 км\n(© участники OpenStreetMap)", fontsize=12)
axes[1].set_axis_off()

fig.tight_layout()
fig.savefig(IMG / "analysis_v0.png", dpi=150, bbox_inches="tight")
print("Сохранено:", IMG / "analysis_v0.png")

In [ ]:
import folium

center = [boundary.to_crs(4326).union_all().centroid.y, boundary.to_crs(4326).union_all().centroid.x]
m = folium.Map(location=center, zoom_start=11, tiles="OpenStreetMap")

def style(feature):
    return {"fillColor": "#1f6feb", "color": "white", "weight": 1, "fillOpacity": 0.15}

folium.GeoJson(
    districts.to_crs(4326).to_json(),
    name="Районы Уфы",
    style_function=style,
    tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Район:"]),
).add_to(m)

for _, r in schools.iterrows():
    folium.CircleMarker(
        location=[r.geometry.y, r.geometry.x], radius=4, color="#1f6feb", fill=True,
        fill_opacity=0.9, popup=f"Школа: {r.get('name', 'без названия')}",
        tooltip=r.get("name", "Школа"),
    ).add_to(m)

for _, r in clinics.iterrows():
    folium.CircleMarker(
        location=[r.geometry.y, r.geometry.x], radius=3, color="#d1242f", fill=True,
        fill_opacity=0.9, popup=f"Поликлиника: {r.get('name', 'без названия')}",
        tooltip=r.get("name", "Поликлиника"),
    ).add_to(m)

folium.LayerControl().add_to(m)
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: white;
     padding: 8px 12px; border: 1px solid #999; font-size: 12px;">
<b>SchoolMed Access, ЛР 2</b><br>
<div style="color:#1f6feb">●</div> школы<br>
<div style="color:#d1242f">●</div> поликлиники<br>
© участники OpenStreetMap (ODbL 1.0)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
m.save(MAPS / "analysis_v0.html")
print("Сохранено:", MAPS / "analysis_v0.html")

## Перенос результатов в репозиторий

Карты из \`docs/img\` и \`docs/maps\` попадают в репозиторий через Pull Request. В Colab ячейка ниже собирает архив \`results_03.zip\`: распакуйте и загрузите файлы в те же папки репозитория в своей ветке. При локальной работе файлы уже лежат в репозитории.

In [ ]:
import shutil
FOLDERS = [IMG, MAPS, PROCESSED]
if IN_COLAB:
    from google.colab import files
    pack = Path("/content/results_03")
    shutil.rmtree(pack, ignore_errors=True)
    for folder in FOLDERS:
        for f in folder.rglob("*"):
            if f.is_file() and f.name != ".gitkeep":
                if f.stat().st_size > 10 * 1024 * 1024:
                    print(f"Пропущен (больше 10 МБ): {f.relative_to(BASE)}")
                    continue
                target = pack / f.relative_to(BASE)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, target)
    archive = shutil.make_archive(str(pack), "zip", pack)
    print("В архиве:", sorted(str(p.relative_to(pack)) for p in pack.rglob("*") if p.is_file()))
    files.download(archive)
else:
    print("Локальный запуск: результаты уже лежат в папках репозитория")

## Выводы

**Вопрос 1. Школы и поликлиники по районам.**

- Всего в Уфе: **163 школы** и **178 поликлиник**; в среднем 1,4 школы и 1,6 поликлиники на 10 тыс. жителей.
- Разброс обеспеченности поликлиниками большой: от **0,5 на 10 тыс.** в Калининском и **0,6** в Орджоникидзевском районах до **2,3–2,4** в Кировском, Советском и Ленинском. Разница — почти в 5 раз.
- Школы распределены ровнее: 1,2–1,7 на 10 тыс. жителей. По плотности на км² выделяется Советский район (1,42 школы/км²), минимальная — в Калининском (0,13).

**Вопрос 2. Удалённость школ от поликлиник.**

- Медианное расстояние от школы до ближайшей поликлиники — **341 м**; у **94,5 %** школ поликлиника есть в пределах 1 км.
- Хуже всего обеспечены: **Ленинский район** — 78,6 % школ с поликлиникой в 1 км, **Калининский** — 85,2 %, **Дёмский** — 92,3 %. В Октябрьском, Орджоникидзевском и Советском — 100 %.
- Самая удалённая школа — № 137 в Калининском районе: **7,9 км** до ближайшей поликлиники.

**Ограничения.**

- Расстояния считаются по прямой, а не по пешеходной сети: реальный путь длиннее, особенно через крупные магистрали.
- Данные OSM неполны: лаборатории и центры исключены (71 объект), часть поликлиник могла быть не нанесена.
- Плотность на 10 тыс. жителей считается по населению районов на 2021 год.
- В слой поликлиник входят частные клиники, поэтому в центральных районах доступность может быть завышена.

**Что это значит для продукта.**

1. Гипотеза о неравномерной доступности подтверждается: разница по поликлиникам на 10 тыс. жителей почти пятикратная.
2. Районы-кандидаты в «проблемные»: Калининский и Орджоникидзевский (мало поликлиник на жителя), Ленинский (низкая доля школ с поликлиникой в 1 км).
3. Для карты и отчёта лучше показывать поликлиники на 10 тыс. жителей, а не абсолютное число объектов: абсолютное число вводит в заблуждение из-за разной площади и населения районов.
4. Следующий шаг — перейти от расстояния по прямой к изохронам по пешеходной сети (ЛР 3–5), это отражено в рисках и бэклоге.

Карты: статичная — \`docs/img/analysis_v0.png\`, интерактивная — \`docs/maps/analysis_v0.html\` (© участники OpenStreetMap, ODbL 1.0).